### New crawler

In [ ]:
import requests
import pandas as pd
import time
import random
import re
import json
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed

class CourseraFinalScraper:
    def __init__(self):
        self.base_api_url = "https://api.coursera.org/api/courses.v1"
        self.filename = "coursera_data_fixed.csv"
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
            "Accept-Language": "en-US,en;q=0.9"
        }
        self.session = requests.Session()
        self.session.headers.update(self.headers)

    def fetch_course_index(self):
        print(f"🚀 [Step 1/2] Fetching first {limit} courses from API...")
        all_courses = []
        start = 0
        batch_size = 100 
        
        # Adding multiple potential difficulty fields for compatibility
        fields = "name,slug,workload,primaryLanguages,difficultyLevel,difficulty,domainTypes,partnerIds,courseType"
        includes = "partnerIds"

        while len(all_courses) < limit:
            params = {
                "start": start,
                "limit": min(batch_size, limit - len(all_courses)),
                "fields": fields,
                "includes": includes
            }
            
            try:
                response = self.session.get(self.base_api_url, params=params, timeout=15)
                if response.status_code != 200: break
                
                data = response.json()
                elements = data.get('elements', [])
                if not elements: break

                partners_map = {p['id']: p['name'] for p in data.get('linked', {}).get('partners.v1', [])}

                for item in elements:
                    # Try both difficultyLevel and difficulty
                    diff = item.get("difficultyLevel") or item.get("difficulty", "N/A")
                    
                    course_data = {
                        "Course title": item.get("name"),
                        "Course_Type": item.get("courseType", "Course"),
                        "Organization": partners_map.get(item.get('partnerIds', [None])[0], "Coursera"),
                        "Workload": item.get("workload", "N/A"),
                        "Difficulty_level": diff, # Might still be N/A, will fix in Step 2
                        "Language": ",".join(item.get("primaryLanguages", [])),
                        "Course_link": f"https://www.coursera.org/learn/{item.get('slug')}",
                        "Ratings": "N/A",
                        "Review count": 0
                    }
                    all_courses.append(course_data)
                
                start += batch_size
                print(f"   Indexed {len(all_courses)} courses...", end='\r')
                time.sleep(0.5)
            except Exception: break
        return all_courses

    def scrape_detailed_stats(self, course_data):
        url = course_data["Course_link"]
        try:
            time.sleep(random.uniform(0.8, 2.0)) # Polite delay
            response = self.session.get(url, timeout=15)
            if response.status_code == 200:
                content = response.text
                
                # 1. Extract Ratings & Reviews
                rating_match = re.search(r'"ratingValue"\s*:\s*([\d\.]+)', content)
                review_match = re.search(r'"reviewCount"\s*:\s*(\d+)', content)
                if rating_match: course_data["Ratings"] = round(float(rating_match.group(1)), 1)
                if review_match: course_data["Review count"] = int(review_match.group(1))

                # 2. FIX: Extract Difficulty Level from HTML if API failed
                # Coursera often puts difficulty in a "difficultyLevel" field inside JSON blobs
                if course_data["Difficulty_level"] == "N/A":
                    # Look for common difficulty keywords in the source code
                    diff_match = re.search(r'"difficultyLevel"\s*:\s*"([^"]+)"', content)
                    if diff_match:
                        course_data["Difficulty_level"] = diff_match.group(1).capitalize()
                    else:
                        # Fallback: Look for UI labels (Beginner, Intermediate, Advanced, Mixed)
                        for level in ["Beginner", "Intermediate", "Advanced", "Mixed"]:
                            if level in content:
                                # Double check it's likely a label by looking for "Level" nearby
                                if re.search(rf'{level}\s*Level', content, re.IGNORECASE):
                                    course_data["Difficulty_level"] = level
                                    break

        except Exception: pass
        return course_data

    def run(self):
        courses = self.fetch_course_index()
        print("\n🚀 [Step 2/2] Extracting Stats & Difficulty via Regex...")
        enriched_data = []
        
        with ThreadPoolExecutor(max_workers=5) as executor:
            future_to_course = {executor.submit(self.scrape_detailed_stats, c): c for c in courses}
            processed = 0
            for future in as_completed(future_to_course):
                enriched_data.append(future.result())
                processed += 1
                if processed % 10 == 0:
                    print(f"   Progress: {processed}/{len(courses)} processed...", end='\r')

        df = pd.DataFrame(enriched_data)
        cols = ["Course title", "Course_Type", "Organization", "Ratings", "Review count", 
                "Difficulty_level", "Workload", "Language", "Course_link"]
        df = df[[c for c in cols if c in df.columns]]
        df.to_csv(self.filename, index=False, encoding='utf_8_sig')
        print(f"\n\n✅ Done! Saved to {self.filename}")

if __name__ == "__main__":
    scraper = CourseraFinalScraper()
    scraper.run()

🚀 [Step 1/2] Fetching first 50 courses from API...
   Indexed 50 courses...
🚀 [Step 2/2] Extracting Stats & Difficulty via Regex...
   Progress: 50/50 processed...

✅ Done! Saved to coursera_data_fixed.csv


In [4]:
# After creating your DataFrame 'df'
print("\n" + "="*30)
print("📊 DATASET SUMMARY")
print("="*30)
df = pd.read_csv("coursera_data_fixed.csv")  # Load the saved CSV to analyze
# 1. Check for Missing Values (NaN)
missing_values = df.isnull().sum()

# 2. Check for Unique Categories (Cardinality)
unique_counts = df.nunique()

# 3. Check Data Types
data_types = df.dtypes

# Combine into a summary table
summary_df = pd.DataFrame({
    "Missing Values": missing_values,
    "Unique Categories": unique_counts,
    "Data Type": data_types
})

print(summary_df)
print("="*30)


📊 DATASET SUMMARY
                  Missing Values  Unique Categories Data Type
Course title                   0                 50    object
Course_Type                    0                  1    object
Organization                   0                 25    object
Ratings                       28                 10   float64
Review count                   0                 27     int64
Difficulty_level               9                  3    object
Workload                      15                 31    object
Language                       0                  5    object
Course_link                    0                 50    object


In [ ]:

print("\n" + "="*50)
print("详细数值分布统计 (Value Counts per Column)")
print("="*50)
target_cols = ["Course_Type", "Difficulty_level", "Ratings", "Language", "Organization"]

for col in target_cols:
    if col in df.columns:
        print(f"\n--- {col} ---")
        print(df[col].value_counts())
    else:
        print(f"\n--- {col} (Column not found) ---")

print("\n" + "="*50)


详细数值分布统计 (Value Counts per Column)

--- Course_Type ---
Course_Type
v2.ondemand    50
Name: count, dtype: int64

--- Difficulty_level ---
Difficulty_level
Beginner        25
Intermediate    15
Advanced         1
Name: count, dtype: int64

--- Ratings ---
Ratings
4.8    6
4.7    5
4.6    3
5.0    2
3.8    1
4.1    1
4.4    1
4.5    1
4.2    1
4.9    1
Name: count, dtype: int64

--- Language ---
Language
en    40
fr     3
ar     3
es     3
ru     1
Name: count, dtype: int64

--- Organization ---
Organization
Coursera                                              12
Google Cloud                                           7
EDUCBA                                                 3
University of Colorado System                          2
University of Pennsylvania                             2
University of Illinois Urbana-Champaign                2
Pearson                                                2
Packt                                                  2
SkillUp                        

In [6]:
import requests
import pandas as pd
import time
import random
import re
import json
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed

class CourseraFullScraper:
    def __init__(self):
        self.base_api_url = "https://api.coursera.org/api/courses.v1"
        self.filename = "coursera_full_database.csv"
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
            "Accept-Language": "en-US,en;q=0.9"
        }
        self.session = requests.Session()
        self.session.headers.update(self.headers)

    def fetch_course_index(self):
        """Step 1: Fetch ALL courses by iterating through the API pagination."""
        print("🚀 [Step 1/2] Fetching complete course index from API...")
        all_courses = []
        start = 0
        batch_size = 100 
        
        fields = "name,slug,workload,primaryLanguages,difficultyLevel,difficulty,domainTypes,partnerIds,courseType"
        includes = "partnerIds"

        while True: # Removed the 'limit' constraint
            params = {
                "start": start,
                "limit": batch_size,
                "fields": fields,
                "includes": includes
            }
            
            try:
                response = self.session.get(self.base_api_url, params=params, timeout=15)
                if response.status_code != 200: 
                    print(f"\n⚠️ API stopped responding at index {start} (Status: {response.status_code})")
                    break
                
                data = response.json()
                elements = data.get('elements', [])
                
                # The exit condition: if the 'elements' list is empty, we reached the end
                if not elements: 
                    break

                partners_map = {p['id']: p['name'] for p in data.get('linked', {}).get('partners.v1', [])}

                for item in elements:
                    diff = item.get("difficultyLevel") or item.get("difficulty", "N/A")
                    
                    course_data = {
                        "Course title": item.get("name"),
                        "Course_Type": item.get("courseType", "Course"),
                        "Organization": partners_map.get(item.get('partnerIds', [None])[0], "Coursera"),
                        "Workload": item.get("workload", "N/A"),
                        "Difficulty_level": diff,
                        "Language": ",".join(item.get("primaryLanguages", [])),
                        "Course_link": f"https://www.coursera.org/learn/{item.get('slug')}",
                        "Ratings": "N/A",
                        "Review count": 0
                    }
                    all_courses.append(course_data)
                
                start += batch_size
                print(f"   Indexed {len(all_courses)} courses...", end='\r')
                time.sleep(0.4) # Slight delay to be polite
                
            except Exception as e: 
                print(f"\n❌ Error during API sync: {e}")
                break
                
        print(f"\n✅ Indexing complete. Total courses found: {len(all_courses)}")
        return all_courses

    def scrape_detailed_stats(self, course_data):
        """Step 2: Scrape details for each course."""
        url = course_data["Course_link"]
        try:
            time.sleep(random.uniform(0.5, 1.2)) # Random delay to prevent IP block
            response = self.session.get(url, timeout=15)
            if response.status_code == 200:
                content = response.text
                
                # Regex for Ratings/Reviews
                rating_match = re.search(r'"ratingValue"\s*:\s*([\d\.]+)', content)
                review_match = re.search(r'"reviewCount"\s*:\s*(\d+)', content)
                if rating_match: course_data["Ratings"] = round(float(rating_match.group(1)), 1)
                if review_match: course_data["Review count"] = int(review_match.group(1))

                # Regex for Difficulty fallback
                if course_data["Difficulty_level"] == "N/A":
                    diff_match = re.search(r'"difficultyLevel"\s*:\s*"([^"]+)"', content)
                    if diff_match:
                        course_data["Difficulty_level"] = diff_match.group(1).capitalize()
                    else:
                        for level in ["Beginner", "Intermediate", "Advanced", "Mixed"]:
                            if re.search(rf'{level}\s*Level', content, re.IGNORECASE):
                                course_data["Difficulty_level"] = level
                                break
        except Exception: pass
        return course_data

    def run(self):
        # 1. Get the list
        courses = self.fetch_course_index()
        
        # 2. Enrich the data
        print("\n🚀 [Step 2/2] Scrapping detailed data (This will take a while)...")
        enriched_data = []
        
        # Max workers kept at 5 to avoid triggering anti-bot protection
        with ThreadPoolExecutor(max_workers=5) as executor:
            future_to_course = {executor.submit(self.scrape_detailed_stats, c): c for c in courses}
            processed = 0
            total = len(courses)
            for future in as_completed(future_to_course):
                enriched_data.append(future.result())
                processed += 1
                if processed % 20 == 0:
                    print(f"   Progress: {processed}/{total} processed...", end='\r')

        df = pd.DataFrame(enriched_data)        
        df_for_stats = df.replace("N/A", pd.NA)
        self.print_summary(df_for_stats)
        df.to_csv(self.filename, index=False, encoding='utf_8_sig')
        print(f"\n\n✅ Done! Data saved to {self.filename}")

    def print_summary(self, df):
        """Displays missing values and value distributions."""
        print("\n" + "="*60)
        print("📊 DATASET ANALYSIS")
        print("="*60)
        
        # Missing values and unique counts
        summary = pd.DataFrame({
            'Missing (Null)': df.isnull().sum(),
            'Unique Values': df.nunique()
        })
        print(summary)
        
        # Value distribution for key columns
        cols_to_check = ["Course_Type", "Difficulty_level", "Language"]
        for col in cols_to_check:
            print(f"\n--- Distribution for: {col} ---")
            print(df[col].value_counts().head(10)) # Shows top 10 categories
        
        print("="*60)

if __name__ == "__main__":
    scraper = CourseraFullScraper()
    scraper.run()
    

🚀 [Step 1/2] Fetching complete course index from API...
   Indexed 18902 courses...
✅ Indexing complete. Total courses found: 18902

🚀 [Step 2/2] Scrapping detailed data (This will take a while)...
   Progress: 18900/18902 processed...
📊 DATASET ANALYSIS
                  Missing (Null)  Unique Values
Course title                   0          18793
Course_Type                    0              2
Organization                   0            374
Workload                       0           3664
Difficulty_level            2161              3
Language                       0             26
Course_link                    0          18902
Ratings                     9850             21
Review count                   0           1812

--- Distribution for: Course_Type ---
Course_Type
v2.ondemand    18782
v2.capstone      120
Name: count, dtype: int64

--- Distribution for: Difficulty_level ---
Difficulty_level
Beginner        9823
Intermediate    6198
Advanced         720
Name: count, dtype: in

In [7]:
df = pd.read_csv("coursera_full_database.csv")
print(df['Course_Type'].value_counts())

Course_Type
v2.ondemand    18782
v2.capstone      120
Name: count, dtype: int64


In [8]:
import requests
import pandas as pd
import time
import random
import re
import json
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed

class CourseraFlexibleScraper:
    def __init__(self):
        self.base_api_url = "https://api.coursera.org/api/courses.v1"
        self.filename = "coursera_full_database_2026.csv"
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        }
        self.session = requests.Session()
        
        # Mapping technical codes to human-readable names
        self.type_map = {
            "v2.ondemand": "Individual Course",
            "v2.capstone": "Capstone Project",
            "v2.specialization": "Specialization",
            "v2.professional-certificate": "Professional Certificate"
        }

    def fetch_course_index(self, limit=None):
        """
        Fetches the master list. 
        If limit is None, it scrapes everything until the API is empty.
        """
        target_str = f"{limit} courses" if limit else "the ENTIRE database"
        print(f"🚀 [Step 1/2] Fetching {target_str} from API...")
        
        all_courses = []
        start = 0
        batch_size = 100 
        fields = "name,slug,workload,primaryLanguages,difficultyLevel,courseType,partnerIds"

        while True:
            # Stop if we reached the user-defined limit
            if limit and len(all_courses) >= limit:
                break

            params = {
                "start": start,
                "limit": min(batch_size, limit - len(all_courses)) if limit else batch_size,
                "fields": fields,
            }
            
            try:
                response = self.session.get(self.base_api_url, params=params, timeout=15)
                if response.status_code != 200: break
                
                data = response.json()
                elements = data.get('elements', [])
                if not elements: break # No more courses available

                for item in elements:
                    raw_type = item.get("courseType", "Course")
                    
                    # Logic to identify payment nature
                    if "degree" in item.get('name', '').lower() or raw_type == "v2.degree":
                        payment_status = "Paid (Degree)"
                    elif raw_type in ["v2.specialization", "v2.professional-certificate", "v2.capstone"]:
                        payment_status = "Subscription (Monthly)"
                    else:
                        payment_status = "Free Audit / Pay for Cert"

                    all_courses.append({
                        "Course title": item.get("name"),
                        "Course_Type": self.type_map.get(raw_type, raw_type),
                        "Payment_Model": payment_status,
                        "Difficulty": item.get("difficultyLevel", "N/A"),
                        "Language": ",".join(item.get("primaryLanguages", [])),
                        "Course_link": f"https://www.coursera.org/learn/{item.get('slug')}",
                        "Ratings": "N/A",
                        "Review count": 0,
                        "Financial_Aid": "Unknown"
                    })
                
                start += batch_size
                print(f"   Indexed {len(all_courses)} courses...", end='\r')
                time.sleep(0.3)
            except Exception as e:
                print(f"\n❌ API Error: {e}")
                break
        
        return all_courses

    def scrape_detailed_stats(self, course_data):
        """Extracts ratings, reviews, and aid info from course pages."""
        url = course_data["Course_link"]
        try:
            time.sleep(random.uniform(0.4, 0.9)) # Speeding up slightly for huge crawls
            response = self.session.get(url, timeout=15)
            if response.status_code == 200:
                content = response.text
                
                # Regex for Ratings & Reviews
                r_match = re.search(r'"ratingValue"\s*:\s*([\d\.]+)', content)
                c_match = re.search(r'"reviewCount"\s*:\s*(\d+)', content)
                if r_match: course_data["Ratings"] = round(float(r_match.group(1)), 1)
                if c_match: course_data["Review count"] = int(c_match.group(1))

                # Financial Aid status check
                course_data["Financial_Aid"] = "Yes" if "Financial aid available" in content else "No"

                # Fix Difficulty fallback
                if course_data["Difficulty"] == "N/A":
                    d_match = re.search(r'"difficultyLevel"\s*:\s*"([^"]+)"', content)
                    if d_match: course_data["Difficulty"] = d_match.group(1).capitalize()

        except Exception: pass
        return course_data

    def run(self, limit=None):
        # 1. Start fetching
        courses = self.fetch_course_index(limit=limit)
        
        print("\n🚀 [Step 2/2] Enriching details (Ratings, Aid, Difficulty Fix)...")
        enriched_data = []
        
        # Max workers at 5 to prevent IP ban on huge datasets
        with ThreadPoolExecutor(max_workers=5) as executor:
            future_to_course = {executor.submit(self.scrape_detailed_stats, c): c for c in courses}
            processed = 0
            total = len(courses)
            for future in as_completed(future_to_course):
                enriched_data.append(future.result())
                processed += 1
                if processed % 50 == 0:
                    print(f"   Progress: {processed}/{total} processed...", end='\r')

        # 3. Save & Summary
        df = pd.DataFrame(enriched_data)
        df.to_csv(self.filename, index=False, encoding='utf_8_sig')
        
        print("\n" + "="*50)
        print("📈 DATA COLLECTION SUMMARY")
        print(f"Total Processed: {len(df)}")
        print("-" * 30)
        print("Payment Model Distribution:")
        print(df["Payment_Model"].value_counts())
        print("="*50)
        print(f"✅ Data saved to {self.filename}")


In [ ]:

scraper = CourseraFlexibleScraper()
scraper.run(limit=50)   # Quick test
#scraper.run(limit=None)   # Full database crawl (~18k courses)
    

🚀 [Step 1/2] Fetching 50 courses from API...
   Indexed 50 courses...
🚀 [Step 2/2] Enriching details (Ratings, Aid, Difficulty Fix)...
